# Train PatchCore on screwit data

Trains a PatchCore anomaly-detection model with anomalib, logs to W&B, exports to
ONNX, and uploads the result to the HF Hub repo configured in `src/shared/config.py`.
All parameters below come from `TrainingSettings`/`Settings` (see `.env.example`) -
nothing here is hardcoded.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable regardless of whether the kernel's cwd is the repo
# root or this notebook's own directory (Jupyter's usual default).
_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from anomalib.data import Folder
from anomalib.deploy import ExportType
from anomalib.engine import Engine
from anomalib.loggers import AnomalibWandbLogger
from anomalib.models import Patchcore
from lightning.pytorch.callbacks import TQDMProgressBar

from src.shared.config import get_settings, get_training_settings

settings = get_settings()
training_settings = get_training_settings()

In [ ]:
# Silence noisy tqdm output in this notebook: the mininterval patch caps how
# often any bar refreshes, and the k_center_greedy patch fully disables
# PatchCore's per-patch "Selecting Coreset Indices." bar - a nested bar under
# Lightning's own progress bar, and the biggest source of redundant redraws.
from functools import partial, partialmethod

from anomalib.models.components.sampling import k_center_greedy
from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, mininterval=1.0)
k_center_greedy.tqdm = partial(k_center_greedy.tqdm, disable=True)

## Dataset

Expects the MVTec-style layout produced by `get_data.sh`:
`train/good`, `test/good` + `test/<defect>`, `ground_truth/<defect>`.

In [ ]:
root = training_settings.data_root / training_settings.category
defect_dirs = sorted(p.name for p in (root / "test").iterdir() if p.is_dir() and p.name != "good")

datamodule = Folder(
    name=training_settings.category,
    root=root,
    normal_dir="train/good",
    abnormal_dir=[f"test/{d}" for d in defect_dirs],
    normal_test_dir="test/good",
    mask_dir=[f"ground_truth/{d}" for d in defect_dirs],
    train_batch_size=training_settings.train_batch_size,
    eval_batch_size=training_settings.eval_batch_size,
)

## Model, training, export

In [ ]:
model = Patchcore(
    num_neighbors=training_settings.num_neighbors,
    coreset_sampling_ratio=training_settings.coreset_sampling_ratio,
)

In [ ]:
wandb_logger = AnomalibWandbLogger(
    project=training_settings.wandb_project,
    name=training_settings.wandb_run_name,
)

# callbacks=[TQDMProgressBar()] forces the plain tqdm progress bar instead of
# Lightning's default RichProgressBar, which collides with anomalib's own
# tqdm-based coreset-selection progress bar and causes a RecursionError.
engine = Engine(
    callbacks=[TQDMProgressBar()],
    logger=wandb_logger,
)

engine.fit(datamodule=datamodule, model=model)
test_results = engine.test(datamodule=datamodule, model=model)

exported_model_path = engine.export(model=model, export_type=ExportType.ONNX)
print(f"Exported to {exported_model_path}")

## Log test predictions to W&B

anomalib's automatic visualization callback isn't wired up in this anomalib
version, so predictions are generated and logged explicitly here.

In [ ]:
import wandb
from anomalib import TaskType
from anomalib.utils.visualization import ImageVisualizer
from anomalib.utils.visualization.image import VisualizationMode

predictions = engine.predict(datamodule=datamodule, model=model)

visualizer = ImageVisualizer(mode=VisualizationMode.FULL, task=TaskType.SEGMENTATION)

wandb_images = []
for batch in predictions:
    for result in visualizer.generate(outputs=batch):
        wandb_images.append(wandb.Image(result.image, caption=str(result.file_name)))

wandb_logger.experiment.log({"test_predictions": wandb_images})
print(f"Logged {len(wandb_images)} test images to W&B.")

## Upload to Hugging Face Hub

Uploads to the same `HF_MODEL_REPO_ID`/`HF_MODEL_FILENAME` the API downloads
from at startup - one config value, not a hardcoded repo id in two places.

In [ ]:
from huggingface_hub import HfApi

HfApi().upload_file(
    path_or_fileobj=str(exported_model_path),
    path_in_repo=settings.hf_model_filename,
    repo_id=settings.hf_model_repo_id,
    token=settings.hf_api_key,
)

## Sanity check: run the exported ONNX model directly

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image

sess = ort.InferenceSession(str(exported_model_path))

img = Image.open(root / "test" / "good" / "000.png").convert("RGB")
arr = np.array(img).astype(np.float32) / 255.0  # HWC, [0, 1]
arr = arr.transpose(2, 0, 1)[None, ...]  # -> NCHW, add batch dim

pred_score, pred_label, anomaly_map, pred_mask = sess.run(None, {"input": arr})
pred_score, pred_label